# Solar Power Generation Forecasting

## Project Overview

Forecasts **daily solar power generation** (MWh) over a 14-day horizon. Synthetic data spans 2 years with strong seasonal patterns (summer highs, winter lows), weather variability, and slight capacity growth.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily solar generation, predict the next 14 days. Solar forecasting is essential for grid integration, storage scheduling, and balancing conventional generation.

## Why This Project Matters

Solar generation is weather-dependent and variable. Accurate forecasts reduce curtailment, optimize battery dispatch, and lower balancing costs. As solar penetration grows, forecast accuracy becomes critical for grid stability.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily solar generation
- Strong seasonal cycle (high summer, low winter)
- Day-to-day weather variability (clouds)
- Slight upward trend (capacity additions)
- No weekly pattern (sun doesn't know weekdays)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `solar_mwh` | Daily solar energy generation (MWh) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'solar_mwh'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Strong seasonal: summer high, winter low
seasonal = 500 + 300 * np.sin(2 * np.pi * (t - 80) / 365)
trend = 0.2 * t  # capacity growth

# Weather variability (cloud factor)
cloud_factor = np.random.beta(5, 2, N_POINTS)  # mostly clear, occasional cloudy
noise = np.random.normal(0, 30, N_POINTS)

solar_mwh = (seasonal + trend) * cloud_factor + noise
solar_mwh = np.maximum(solar_mwh, 10).round(1)

df = pd.DataFrame({'date': dates, 'solar_mwh': solar_mwh})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,solar_mwh
0,2022-01-01,183.7
1,2022-01-02,197.3
2,2022-01-03,170.2
3,2022-01-04,148.5
4,2022-01-05,221.4


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('solar_mwh Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("solar_mwh Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

solar_mwh Statistics:
count    730.000000
mean     410.292055
std      180.339499
min       76.400000
25%      259.575000
50%      392.900000
75%      552.400000
max      861.500000
Name: solar_mwh, dtype: float64

CV: 0.440
Skewness: 0.292


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       69.3 | RMSE:       81.3 | MAPE: 34.13%
Seasonal Naive (7)             | MAE:       77.7 | RMSE:       94.5 | MAPE: 43.24%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared        RMSE  Time Taken
Model                                                                           
KernelRidge                      2254.948379 -172.380645  482.108474    0.047841
DummyRegressor                    231.349755  -16.719212  154.122718    0.014229
QuantileRegressor                 198.008400  -14.154492  142.532911    0.080173
GaussianProcessRegressor          156.068730  -10.928364  126.454677    0.094678
LinearSVR                          92.794878   -6.061144   97.293092    0.009994
DecisionTreeRegressor              92.652436   -6.050187   97.217576    0.028506
NuSVR                              92.288055   -6.022158   97.024131    0.042505
ExtraTreeRegressor                 81.268266   -5.174482   90.979747    0.017355
MLPRegressor                       74.710975   -4.670075   87.184416    0.551699
CatBoostRegressor                  67.552992   -4.119461   82.843144    2.347693


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: extra_tree
FLAML (extra_tree)             | MAE:       35.6 | RMSE:       39.5 | MAPE: 12.81%
FLAML Test (extra_tree)        | MAE:       67.4 | RMSE:       81.4 | MAPE: 39.16%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       67.9 | RMSE:       83.0 | MAPE: 40.43%
SF AutoETS                     | MAE:       68.4 | RMSE:       84.0 | MAPE: 41.07%
SF SeasonalNaive               | MAE:       55.4 | RMSE:       71.4 | MAPE: 33.29%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                  model       MAE      RMSE      MAPE
     FLAML (extra_tree) 35.575033 39.508517 12.810793
       SF SeasonalNaive 55.435714 71.390891 33.294250
FLAML Test (extra_tree) 67.419949 81.419692 39.164598
           SF AutoARIMA 67.945672 82.971201 40.431340
             SF AutoETS 68.401218 84.025166 41.071184
     Naive (Last Value) 69.321429 81.250376 34.128138
     Seasonal Naive (7) 77.735714 94.530672 43.244387

Top 3:
                  model       MAE      RMSE      MAPE
     FLAML (extra_tree) 35.575033 39.508517 12.810793
       SF SeasonalNaive 55.435714 71.390891 33.294250
FLAML Test (extra_tree) 67.419949 81.419692 39.164598


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -18.65, Std: 79.25


## Interpretation and Business Insight

- Solar generation has the strongest seasonal pattern of any energy source
- Day-to-day variability (weather) makes short-term forecasting challenging
- No weekly pattern — purely driven by solar irradiance and cloud cover
- Capacity growth adds a slow upward trend
- Seasonal naive is surprisingly strong due to dominant annual cycle

## Limitations

1. Synthetic — real solar depends on irradiance, cloud cover, panel tilt
2. No weather/irradiance features (the key driver)
3. No hourly profile (solar output is zero at night)
4. Single plant aggregate — real grids have distributed solar

## How to Improve This Project

1. Add solar irradiance and cloud cover forecasts
2. Use hourly data for intraday dispatch
3. Model individual plants with panel specs
4. Add satellite imagery for cloud nowcasting
5. Use Chronos-Bolt for zero-shot solar forecasting

## Production Considerations

- Day-ahead and week-ahead forecasting for grid operators
- Integration with weather forecast APIs
- Real-time satellite-based nowcasting updates
- Battery dispatch optimization using forecasts

## Common Mistakes

1. Expecting weekly patterns in solar data (there are none)
2. Ignoring weather — it explains most day-to-day variance
3. Using annual averages instead of seasonal models
4. Not accounting for panel degradation over time

## Mini Challenge / Exercises

1. Add a synthetic cloud-cover feature and measure improvement
2. Decompose into seasonal + residual and forecast each separately
3. Build a clear-sky model and then model cloud deviation
4. Try predicting monthly totals instead of daily

## Final Summary / Key Takeaways

1. Solar forecasting has strong seasonality but high daily variability
2. Weather (not included) is the dominant short-term driver
3. Seasonal naive is a hard-to-beat baseline
4. Capacity growth must be tracked over time
5. Real deployment needs weather data integration

In [18]:
import json
metrics = {
    'project': 'Solar Power Generation Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Solar Power Generation Forecasting — Complete ===")

Metrics saved.

=== Solar Power Generation Forecasting — Complete ===
